# Survival Analysis — Customer Churn

---

## Project Overview

**Objective:** Understand *when* and *why* customers churn using survival analysis techniques.
Complements the binary classification model in `3a)Data_Preprocessing_Models_Churn` by adding a temporal dimension.

**Data:** `mask_pay` — customers with at least one settlement event (67,440 customers).
Every customer in this population has a known outcome and duration, making censoring a non-issue.

| Variable | Description |
|----------|-------------|
| `TIME_TO_CHURN_D` | Duration of last contract until settlement (days) |
| `IS_CHURN` | Event indicator: 1 = churned, 0 = returned within 30 days (censored) |

---

## Notebook Structure

```
 0. SET UP                  Imports, constants, paths
 1. LOAD DATASET            Load ABT, filter to mask_pay
 2. KAPLAN-MEIER            Overall survival curve + group comparisons
 3. COX PH MODEL            Hazard ratios per feature
 4. SUMMARY                 Key findings
```

---
## 0. SET UP
---

In [2]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

import src.code.io_utils as io_utils

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid', palette='muted')

CLIENT_PATH = io_utils.output_path('prepared/abt.parquet')

---
## 1. LOAD DATASET
---

In [3]:
abt = io_utils.load(CLIENT_PATH)

# Filter to customers with settlement history (training population)
mask_pay = abt[abt[["EVER_SAN", "EVER_SOL", "EVER_RBT"]].eq(1).any(axis=1)].copy()

print(f"Total ABT:   {len(abt):,}")
print(f"mask_pay:    {len(mask_pay):,}")
print(f"IS_CHURN=1:  {mask_pay['IS_CHURN'].sum():,}  ({mask_pay['IS_CHURN'].mean():.1%})")
print(f"IS_CHURN=0:  {(mask_pay['IS_CHURN']==0).sum():,}  ({(mask_pay['IS_CHURN']==0).mean():.1%})")
print()
print("TIME_TO_CHURN_D stats (churned only):")
print(mask_pay.loc[mask_pay['IS_CHURN']==1, 'TIME_TO_CHURN_D'].describe().round(1))

ModuleNotFoundError: No module named 'pandas.api.internals'

---
## 2. KAPLAN-MEIER
---

The Kaplan-Meier estimator shows the probability of a customer still being retained at time *t*.
No assumptions about the underlying distribution are required.

In [ ]:
# ── 2.1 Overall survival curve ────────────────────────────────────────────────
kmf = KaplanMeierFitter()
kmf.fit(
    durations=mask_pay["TIME_TO_CHURN_D"],
    event_observed=mask_pay["IS_CHURN"],
    label="All customers"
)

fig, ax = plt.subplots(figsize=(10, 5))
kmf.plot_survival_function(ax=ax, ci_show=True)
ax.set_xlabel("Days since last contract start")
ax.set_ylabel("Retention probability")
ax.set_title("Kaplan-Meier — Customer Retention")

# Annotate median survival time
median_t = kmf.median_survival_time_
ax.axvline(median_t, color='tomato', linestyle='--', alpha=0.7)
ax.text(median_t + 10, 0.52, f'Median: {median_t:.0f} days', color='tomato', fontsize=9)

plt.tight_layout()
plt.show()

print(f"Median survival time: {median_t:.0f} days ({median_t/30.44:.1f} months)")

In [ ]:
# ── 2.2 Group comparisons with log-rank test ──────────────────────────────────
# Change grouping variable as needed — e.g. IS_EARLY_SETTLER, CSP tier, etc.
GROUP_COL = "IS_EARLY_SETTLER"  # 1 = early settler (SAN/RBT), 0 = on-time (SOL)

fig, ax = plt.subplots(figsize=(10, 5))

groups = mask_pay[GROUP_COL].unique()
kmf_list = []
for g in sorted(groups):
    subset = mask_pay[mask_pay[GROUP_COL] == g]
    kmf_g = KaplanMeierFitter()
    kmf_g.fit(
        durations=subset["TIME_TO_CHURN_D"],
        event_observed=subset["IS_CHURN"],
        label=f"{GROUP_COL}={g}  (n={len(subset):,})"
    )
    kmf_g.plot_survival_function(ax=ax, ci_show=True)
    kmf_list.append((g, subset))

# Log-rank test between the two groups
if len(kmf_list) == 2:
    g0_data = kmf_list[0][1]
    g1_data = kmf_list[1][1]
    result = logrank_test(
        g0_data["TIME_TO_CHURN_D"], g1_data["TIME_TO_CHURN_D"],
        event_observed_A=g0_data["IS_CHURN"],
        event_observed_B=g1_data["IS_CHURN"]
    )
    ax.set_title(f"KM by {GROUP_COL}  |  Log-rank p={result.p_value:.4f}")
else:
    ax.set_title(f"KM by {GROUP_COL}")

ax.set_xlabel("Days since last contract start")
ax.set_ylabel("Retention probability")
plt.tight_layout()
plt.show()

---
## 3. COX PROPORTIONAL HAZARDS MODEL
---

Cox PH estimates the effect of each feature on the *hazard* (instantaneous churn risk).
A hazard ratio > 1 means higher churn risk; < 1 means lower risk.

In [ ]:
# ── 3.1 Prepare features ──────────────────────────────────────────────────────
# Select numeric, non-leaky features available at contract start
# TIME_TO_CHURN_D and IS_CHURN are the duration/event columns — not features
LEAKY = [
    "IS_CHURN", "TIME_TO_CHURN_D",
    "EVER_SAN", "EVER_SOL", "EVER_RBT", "N_SAN", "N_SOL", "N_RBT",
    "IS_EARLY_SETTLER", "IS_SAN",
    "LAST_OBS_DATE_SOL", "LAST_OBS_DATE_SAN", "LAST_OBS_DATE_RBT",
    "TOTAL_CRD", "TOTAL_SREC", "TOTAL_RN", "TOTAL_RD",
    "LAST_RISK", "MAX_RISKA", "LAST_DPOS", "LAST_DATFIN", "FIRST_D1FIN",
    "FIRST_DCREAT", "LAST_DCREAT", "CONTRIB",
]

cox_features = [
    c for c in mask_pay.select_dtypes(include='number').columns
    if c not in LEAKY
]

cox_df = mask_pay[cox_features + ["TIME_TO_CHURN_D", "IS_CHURN"]].copy()

# Drop columns with too many nulls (Cox doesn't handle NaN)
cox_df = cox_df.dropna(thresh=int(len(cox_df) * 0.5), axis=1)
cox_df = cox_df.fillna(cox_df.median(numeric_only=True))

# Remove zero-variance columns
cox_df = cox_df.loc[:, cox_df.std() > 0]

print(f"Features for Cox model: {len(cox_df.columns) - 2}")
print(cox_df.shape)

In [ ]:
# ── 3.2 Fit Cox PH model ──────────────────────────────────────────────────────
cph = CoxPHFitter(penalizer=0.1)  # L2 regularization to handle correlated features
cph.fit(
    cox_df,
    duration_col="TIME_TO_CHURN_D",
    event_col="IS_CHURN",
    show_progress=True
)

cph.print_summary()

In [ ]:
# ── 3.3 Hazard ratio plot (top 20 features) ───────────────────────────────────
hr = cph.summary[["exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%", "p"]].copy()
hr = hr.sort_values("exp(coef)", ascending=False).head(20)

fig, ax = plt.subplots(figsize=(9, 7))
y_pos = range(len(hr))

ax.barh(y_pos, hr["exp(coef)"] - 1,
        xerr=[hr["exp(coef)"] - hr["exp(coef) lower 95%"],
              hr["exp(coef) upper 95%"] - hr["exp(coef)"]],
        color=["tomato" if v > 1 else "steelblue" for v in hr["exp(coef)"]],
        edgecolor="white", height=0.6)

ax.axvline(0, color='black', linewidth=0.8)
ax.set_yticks(y_pos)
ax.set_yticklabels(hr.index, fontsize=9)
ax.set_xlabel("Hazard Ratio - 1  (red = higher churn risk, blue = lower)")
ax.set_title("Cox PH — Top 20 Features by Hazard Ratio")
plt.tight_layout()
plt.show()

---
## 4. SUMMARY
---

In [ ]:
print("=" * 50)
print("SURVIVAL ANALYSIS SUMMARY")
print("=" * 50)
print(f"Population:          {len(mask_pay):,} customers (mask_pay)")
print(f"Churn events:        {mask_pay['IS_CHURN'].sum():,} ({mask_pay['IS_CHURN'].mean():.1%})")
print(f"Median survival:     {kmf.median_survival_time_:.0f} days"
      f" ({kmf.median_survival_time_/30.44:.1f} months)")
print()
print("Top 5 features increasing churn risk (HR > 1):")
top_risk = cph.summary["exp(coef)"].sort_values(ascending=False).head(5)
for feat, hr_val in top_risk.items():
    print(f"  {feat:<30} HR = {hr_val:.3f}")
print()
print("Top 5 features decreasing churn risk (HR < 1):")
top_protect = cph.summary["exp(coef)"].sort_values().head(5)
for feat, hr_val in top_protect.items():
    print(f"  {feat:<30} HR = {hr_val:.3f}")